# Program 11: Summarising long text with a pre-trained model

**Aim.** Summarise a long passage with a Hugging Face summarisation pipeline, and check what the summary lost.

**Theory.** Abstractive summarisation (BART/DistilBART) generates new sentences conditioned on the input. Because it can drop or alter details, the program checks that the numbers and the medicine name survive. This is the same worry as in MedSaarthi's simplification step. Transformers 5 removed the 'summarization' pipeline named in the manual, so the program loads the same model with its tokenizer and calls generate(), which is what the pipeline did internally.

In [1]:
import sys
sys.path.insert(0, '.')
from common import *


In [2]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
NAME = 'sshleifer/distilbart-cnn-12-6'
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(NAME).eval()
print(type(model).__name__, model.num_parameters() // 1_000_000, 'M parameters')

def summarise(text, max_new=120, min_new=40):
    batch = tok(text, return_tensors='pt', truncation=True, max_length=1024)
    with torch.no_grad():
        ids = model.generate(**batch, num_beams=4, max_new_tokens=max_new, min_new_tokens=min_new, do_sample=False)
    return tok.decode(ids[0], skip_special_tokens=True).strip()
passage = (ROOT / 'data' / 'leaflets' / 'amoxicillin.md').read_text(encoding='utf-8')
passage = passage.split('\n', 3)[3]   # drop the title and the synthetic notice
print(len(passage.split()), 'words in the passage')

BartForConditionalGeneration 305 M parameters
332 words in the passage


In [3]:
out = summarise(passage[:3500])
print(out)
print('\ncompression:', len(out.split()), 'of', len(passage[:3500].split()), 'words')

Amoxicillin is a penicillin-type antibiotic . It treats infections caused by bacteria, for example some chest, ear, throat, urinary, and dental infections . It does not work against viral infections such as the common cold or flu .

compression: 38 of 332 words


In [4]:
import re
nums_src, nums_out = set(re.findall(r'\d+', passage[:3500])), set(re.findall(r'\d+', out))
print('numbers only in the summary (possible invention):', sorted(nums_out - nums_src) or 'none')
print('numbers in the source that the summary dropped:', sorted(nums_src - nums_out)[:10])
print('medicine name kept:', 'amoxicillin' in out.lower())

numbers only in the summary (possible invention): none
numbers in the source that the summary dropped: ['1', '2', '25', '3', '4', '5', '6']
medicine name kept: True


## Observations

The summary is shorter and readable, but the check shows what it dropped. For dosing text MedSaarthi does not summarise: doses, times and days are written by code from verified fields.